<a href="https://colab.research.google.com/github/annatagus/python_aulas_exercicios/blob/main/2_M2_Exercicios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Abaixo encontra a resolução de todos os exercícios da folha do **Módulo 2 (Estatística Descritiva e Análise Exploratória de Dados - AED)**, desenvolvida em Python para nível *beginner* com `pandas`, `numpy`, `scipy` e `seaborn`, incluindo os comentários, análises estatísticas e testes de hipóteses solicitados.

---

## Preparação e Carregamento do Dataset

In [2]:
import numpy as np
import pandas as pd
from scipy import stats
import seaborn as sns

# Carregar o dataset Penguins
df_penguins = sns.load_dataset("penguins")

---

## GRUPO A: ENTENDIMENTO E MANIPULAÇÃO DOS DADOS

### Exercício 1. Entendimento do Negócio e dos Dados

In [3]:
# 1. Primeiras linhas
print("--- Primeiras linhas ---")
print(df_penguins.head())

# 2. Shape, Info e Describe
print("\n--- Dimensões (Shape) ---")
print(df_penguins.shape)

print("\n--- Informação Geral ---")
print(df_penguins.info())

print("\n--- Resumo Estatístico ---")
print(df_penguins.describe())

# 3. Separar colunas numéricas de categóricas
colunas_numericas = df_penguins.select_dtypes(include=[np.number]).columns.tolist()
colunas_categoricas = df_penguins.select_dtypes(include=["object", "category"]).columns.tolist()

print("\nColunas Numéricas:", colunas_numericas)
print("Colunas Categóricas:", colunas_categoricas)

# 4. Valores em falta por coluna e duplicados
print("\n--- Valores em falta por coluna ---")
print(df_penguins.isnull().sum())

print("\nNúmero de linhas duplicadas:", df_penguins.duplicated().sum())

# 5. Entendimento do Negócio:
# - Para prever 'species' (variável categórica/discreta): trata-se de um problema de CLASSIFICAÇÃO.
# - Para prever 'body_mass_g' (variável numérica contínua): trata-se de um problema de REGRESSÃO.

--- Primeiras linhas ---
  species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0  Adelie  Torgersen            39.1           18.7              181.0   
1  Adelie  Torgersen            39.5           17.4              186.0   
2  Adelie  Torgersen            40.3           18.0              195.0   
3  Adelie  Torgersen             NaN            NaN                NaN   
4  Adelie  Torgersen            36.7           19.3              193.0   

   body_mass_g     sex  
0       3750.0    Male  
1       3800.0  Female  
2       3250.0  Female  
3          NaN     NaN  
4       3450.0  Female  

--- Dimensões (Shape) ---
(344, 7)

--- Informação Geral ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    object 
 1   island             344 non-null    object 
 2   bill_length_mm   

---

### Exercício 2. Seleção, Filtragem e Agregação

In [4]:
# 1. Seleção com loc
df_selecao = df_penguins.loc[:, ["species", "island", "body_mass_g"]]
print("--- 1. Seleção com loc ---")
print(df_selecao.head())

# 2. Filtrar Gentoo com body_mass_g > 5000g
mascara_gentoo_pesados = (df_penguins["species"] == "Gentoo") & (df_penguins["body_mass_g"] > 5000)
qtd_gentoo_pesados = mascara_gentoo_pesados.sum()
print(f"\n2. Nº de pinguins Gentoo com peso > 5000g: {qtd_gentoo_pesados}")

# 3. Massa corporal média por espécie (groupby)
massa_media_esp = df_penguins.groupby("species")["body_mass_g"].mean()
print("\n3. Massa corporal média por espécie:")
print(massa_media_esp)

# 4. crosstab de species x island
ct_especie_ilha = pd.crosstab(df_penguins["species"], df_penguins["island"])
print("\n4. Tabela de Cruzamento (crosstab) species x island:")
print(ct_especie_ilha)

# 5. pivot_table de massa média por species (linhas) e sex (colunas)
pv_massa = df_penguins.pivot_table(
    values="body_mass_g", index="species", columns="sex", aggfunc="mean"
)
print("\n5. Tabela Dinâmica (pivot_table) de Massa Média:")
print(pv_massa)

--- 1. Seleção com loc ---
  species     island  body_mass_g
0  Adelie  Torgersen       3750.0
1  Adelie  Torgersen       3800.0
2  Adelie  Torgersen       3250.0
3  Adelie  Torgersen          NaN
4  Adelie  Torgersen       3450.0

2. Nº de pinguins Gentoo com peso > 5000g: 61

3. Massa corporal média por espécie:
species
Adelie       3700.662252
Chinstrap    3733.088235
Gentoo       5076.016260
Name: body_mass_g, dtype: float64

4. Tabela de Cruzamento (crosstab) species x island:
island     Biscoe  Dream  Torgersen
species                            
Adelie         44     56         52
Chinstrap       0     68          0
Gentoo        124      0          0

5. Tabela Dinâmica (pivot_table) de Massa Média:
sex             Female         Male
species                            
Adelie     3368.835616  4043.493151
Chinstrap  3527.205882  3938.970588
Gentoo     4679.741379  5484.836066


---

## GRUPO B: ESTATÍSTICA DESCRITIVA (UNIVARIADA)

### Exercício 3. Medidas, Distribuição e Outliers

In [5]:
# Limpeza prévia de nulos em body_mass_g para cálculos estatísticos
body_mass = df_penguins["body_mass_g"].dropna()

# 1. Describe + Assimetria (Skewness) e Curtose (Kurtosis)
print("--- 1. Medidas de body_mass_g ---")
print(body_mass.describe())
skew_bm = body_mass.skew()
kurt_bm = body_mass.kurtosis()
print(f"Assimetria (Skewness): {skew_bm:.4f}")
print(f"Curtose (Kurtosis): {kurt_bm:.4f}")

# 2. Classificação da distribuição:
# - A assimetria é positiva moderada (skew > 0, rácio média/mediana com ligeira inclinação à direita).
# - Apresenta um formato multimodal no geral devido à mistura das três espécies de tamanhos distintos.

# 3. Teste de Normalidade de Shapiro-Wilk
stat_shapiro, p_val_shapiro = stats.shapiro(body_mass)
print(f"\n3. Teste de Shapiro-Wilk (body_mass_g): Estatística={stat_shapiro:.4f}, p-value={p_val_shapiro:.4e}")
if p_val_shapiro > 0.05:
    print("Conclusão: A distribuição é NORMAL (p > 0.05).")
else:
    print("Conclusão: A distribuição NÃO é NORMAL (p <= 0.05).")

# 4. Outliers: Método IQR vs. Método Z-score (>3)
# Método IQR
q1 = body_mass.quantile(0.25)
q3 = body_mass.quantile(0.75)
iqr = q3 - q1
outliers_iqr = body_mass[(body_mass < (q1 - 1.5 * iqr)) | (body_mass > (q3 + 1.5 * iqr))]

# Método Z-score
z_scores = np.abs(stats.zscore(body_mass))
outliers_z = body_mass[z_scores > 3]

print(f"\n4. Nº de Outliers identificados por IQR: {len(outliers_iqr)}")
print(f"   Nº de Outliers identificados por Z-Score (>3): {len(outliers_z)}")

# 5. Repetir análise para flipper_length_mm
flipper = df_penguins["flipper_length_mm"].dropna()
print("\n--- 5. Comparação com flipper_length_mm ---")
print(f"Flipper Length -> Skewness: {flipper.skew():.4f} | Kurtosis: {flipper.kurtosis():.4f}")
_, p_val_flip = stats.shapiro(flipper)
print(f"Shapiro-Wilk p-value (flipper): {p_val_flip:.4e}")

--- 1. Medidas de body_mass_g ---
count     342.000000
mean     4201.754386
std       801.954536
min      2700.000000
25%      3550.000000
50%      4050.000000
75%      4750.000000
max      6300.000000
Name: body_mass_g, dtype: float64
Assimetria (Skewness): 0.4703
Curtose (Kurtosis): -0.7192

3. Teste de Shapiro-Wilk (body_mass_g): Estatística=0.9592, p-value=3.6790e-08
Conclusão: A distribuição NÃO é NORMAL (p <= 0.05).

4. Nº de Outliers identificados por IQR: 0
   Nº de Outliers identificados por Z-Score (>3): 0

--- 5. Comparação com flipper_length_mm ---
Flipper Length -> Skewness: 0.3457 | Kurtosis: -0.9843
Shapiro-Wilk p-value (flipper): 3.5401e-09


---

### Exercício 4. Univariada Categórica

In [6]:
# 1. Nº de categorias (cardinalidade)
print("--- 1. Cardinalidade ---")
for col in ["species", "island", "sex"]:
    print(f"{col}: {df_penguins[col].nunique()} categorias distintas")

# 2. value_counts (incluindo nulos)
print("\n--- 2. Frequências de 'species' (com nulos) ---")
print(df_penguins["species"].value_counts(dropna=False))

print("\n--- Frequências de 'island' (com nulos) ---")
print(df_penguins["island"].value_counts(dropna=False))

# 3. Comentários:
# - Não existem categorias raríssimas nem extremo desequilíbrio nas espécies (Adelie é a mais numerosa com 152, Gentoo tem 124 e Chinstrap 68).
# - A ilha Biscoe e Dream concentram a grande maioria da população, enquanto Torgersen acolhe apenas pinguins da espécie Adelie.

--- 1. Cardinalidade ---
species: 3 categorias distintas
island: 3 categorias distintas
sex: 2 categorias distintas

--- 2. Frequências de 'species' (com nulos) ---
species
Adelie       152
Gentoo       124
Chinstrap     68
Name: count, dtype: int64

--- Frequências de 'island' (com nulos) ---
island
Biscoe       168
Dream        124
Torgersen     52
Name: count, dtype: int64


---

## GRUPO C: INFERÊNCIA: CONFIANÇA, CORRELAÇÕES E TESTES

### Exercício 5. Intervalos de Confiança e Amostragem

In [7]:
body_mass = df_penguins["body_mass_g"].dropna()

# 1. Intervalo de Confiança a 95% para a média populacional
media_pop = body_mass.mean()
sem_pop = stats.sem(body_mass)  # Erro padrão da média
ic_95 = stats.t.interval(0.95, df=len(body_mass)-1, loc=media_pop, scale=sem_pop)

print(f"1. Média Populacional da Massa: {media_pop:.2f} g")
print(f"   Intervalo de Confiança 95%: [{ic_95[0]:.2f} g, {ic_95[1]:.2f} g]")

# 2. Amostra aleatória de 30% (frac=0.3)
df_amostra = df_penguins.sample(frac=0.3, random_state=42)
media_amostra = df_amostra["body_mass_g"].dropna().mean()

print(f"\n2. Média da Amostra (30% dos dados): {media_amostra:.2f} g")

# 3. Comentários:
# - A amostra de 30% é bastante representativa, já que a média amostral está muito próxima da média populacional e dentro do intervalo de confiança.
# - Se a dimensão da amostra for menor, o erro padrão (SEM) AUMENTA, o que faz com que o Intervalo de Confiança fique MAIS LARGO (menor precisão).

1. Média Populacional da Massa: 4201.75 g
   Intervalo de Confiança 95%: [4116.46 g, 4287.05 g]

2. Média da Amostra (30% dos dados): 4187.75 g


---

### Exercício 6. Correlações e Testes de Hipóteses

In [8]:
# Limpeza para testes quantitativos
df_clean = df_penguins.dropna(subset=["bill_length_mm", "bill_depth_mm", "body_mass_g", "sex"]).copy()

# 1. Correlação de Pearson e Spearman entre bill_length_mm e bill_depth_mm
corr_p, p_p = stats.pearsonr(df_clean["bill_length_mm"], df_clean["bill_depth_mm"])
corr_s, p_s = stats.spearmanr(df_clean["bill_length_mm"], df_clean["bill_depth_mm"])

print(f"1. Correlação de Pearson: {corr_p:.4f} (p-val: {p_p:.4f})")
print(f"   Correlação de Spearman: {corr_s:.4f} (p-val: {p_s:.4f})")

# 2. Matriz de Correlação das 4 variáveis numéricas
col_num = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
matriz_corr = df_clean[col_num].corr()
print("\n2. Matriz de Correlação de Pearson:")
print(matriz_corr)
# Par mais correlacionado: body_mass_g e flipper_length_mm (r ≈ 0.87)

# 3. t-test independente da body_mass_g entre Machos e Fêmeas
massa_machos = df_clean[df_clean["sex"] == "Male"]["body_mass_g"]
massa_femeas = df_clean[df_clean["sex"] == "Female"]["body_mass_g"]

t_stat, p_val_ttest = stats.ttest_ind(massa_machos, massa_femeas)
print(f"\n3. t-test (Machos vs Fêmeas): t-stat={t_stat:.4f}, p-value={p_val_ttest:.4e}")

# 4. ANOVA de body_mass_g entre as 3 espécies
massa_adelie = df_clean[df_clean["species"] == "Adelie"]["body_mass_g"]
massa_chinstrap = df_clean[df_clean["species"] == "Chinstrap"]["body_mass_g"]
massa_gentoo = df_clean[df_clean["species"] == "Gentoo"]["body_mass_g"]

f_stat, p_val_anova = stats.f_oneway(massa_adelie, massa_chinstrap, massa_gentoo)
print(f"\n4. ANOVA entre Espécies: F-stat={f_stat:.4f}, p-value={p_val_anova:.4e}")

# 5. Teste do Qui-Quadrado (Chi2) para species x island
tabela_contingencia = pd.crosstab(df_penguins["species"], df_penguins["island"])
chi2, p_val_chi2, dof, expected = stats.chi2_contingency(tabela_contingencia)
print(f"\n5. Qui-Quadrado (species x island): Chi2={chi2:.4f}, p-value={p_val_chi2:.4e}")

# 6. Três Conclusões Fundamentadas:
# - Conclusão 1 (Dimorfismo Sexual): O t-test confirma com elevada significância estatística (p < 0.001) que os machos têm uma massa corporal média significativamente maior do que as fêmeas.
# - Conclusão 2 (Diferença entre Espécies): A ANOVA prova que existem diferenças altamente significativas na massa corporal entre as espécies (p < 0.001), sendo a espécie Gentoo substancialmente mais pesada.
# - Conclusão 3 (Distribuição Geográfica): O teste do Qui-Quadrado (p < 0.001) demonstra que a distribuição das espécies varia de forma dependente em relação à ilha de habitação (ex: a espécie Gentoo só habita a ilha Biscoe).

1. Correlação de Pearson: -0.2286 (p-val: 0.0000)
   Correlação de Spearman: -0.2139 (p-val: 0.0001)

2. Matriz de Correlação de Pearson:
                   bill_length_mm  bill_depth_mm  flipper_length_mm  \
bill_length_mm           1.000000      -0.228626           0.653096   
bill_depth_mm           -0.228626       1.000000          -0.577792   
flipper_length_mm        0.653096      -0.577792           1.000000   
body_mass_g              0.589451      -0.472016           0.872979   

                   body_mass_g  
bill_length_mm        0.589451  
bill_depth_mm        -0.472016  
flipper_length_mm     0.872979  
body_mass_g           1.000000  

3. t-test (Machos vs Fêmeas): t-stat=8.5417, p-value=4.8972e-16

4. ANOVA entre Espécies: F-stat=341.8949, p-value=3.7445e-81

5. Qui-Quadrado (species x island): Chi2=299.5503, p-value=1.3546e-63
